# 30. 제목 경계·토큰 parent-child·제목 계층 parent-child 비교 — 검색 5:5 고정

**목적:** A 작은 청크 반환, B 토큰 크기 기반 부모 반환, C 원문 제목 계층 기반 부모 반환의 세 구성을 비교합니다.
검색 조건은 사용자와 합의한 E5:Kiwi BM25 = **5:5**, 각 arm 후보 20개, RRF k=60으로 고정합니다.

| 후보 | 검색·임베딩 단위 | 검색 후 반환 문맥 |
|---|---|---|
| A `title_boundary` | 제목 안에서 400-token SentenceSplitter, overlap 48 | 검색된 작은 청크 그대로 |
| B `parent_child` | 제목 안에 1,200-token 부모를 만들고, 그 안의 400-token 자식으로 검색 | 상위 자식의 부모를 가져오고 동일 부모는 한 번만 반환 |
| C `heading_parent_child` | XML SECTION/TITLE/SUBTITLE 계층의 제목 구간. E5 512-token 한도를 넘는 구간만 400-token 검색 조각으로 추가 분할 | 자식 제목 구간은 실제 상위 제목 전체. 하위 제목이 있는 내부 노드의 직접 본문은 그 내부 노드 전체 |

세 후보 모두 제목 경계를 지킵니다. C는 SECTION 중첩 및 TITLE/SUBTITLE 순서를 사용하며 LLM이나 번호 정규식으로 제목 수준을 추측하지 않습니다. B는 LlamaIndex **HierarchicalNodeParser**를 사용하며 부모·자식 ID 연결을 실제로 생성합니다.
B는 부모 분할 뒤 자식을 나누므로 A와 자식 경계도 달라질 수 있습니다. 따라서 **계층 분할 + 부모 확장의 구성 비교**이며, 부모 확장만의 독립 효과는 아닙니다.
부모 1,200/자식 400은 이번 첫 비교 설정이지, 모든 크기를 최적화한 최종값이 아닙니다. AutoMergingRetriever의 임계값 기반 병합은 사용하지 않습니다.

**평가 범위:** 29번과 같은 5개 회사·2024 사업보고서와 기존 질문 25개, 기존 정답 근거를 묶은 복합 질문 5개.
총 30개 개발 질문이며 독립 holdout이 아닙니다. 단일/복합 결과를 분리합니다. 복합 질문의 근거 목록은 검색 전에 고정합니다.

**공정성:** 상위 **자식/작은 청크 5개**까지 검색하고 B/C는 그 부모를 중복 제거합니다. 부모 5개가 나올 때까지 더 검색하지 않습니다. C의 원래 부모가 2,400토큰보다 크면 공통 예산에 담지 못할 수 있습니다. 이 경우 `first_context_over_budget`으로 구분하며 검색 실패나 제목 계층의 품질 실패로 단정하지 않습니다.
원래 반환 문맥과 **공통 2,400 E5-token 예산**의 반환 문맥을 따로 평가합니다. 예산은 비교용이며 최종 운영 제한이 아닙니다.
앞에서부터 문맥 단위를 통째로 넣고, 다음 단위가 예산을 넘으면 멈춥니다. 문장 중간을 잘라 정답을 훼손하지 않습니다.

**판단 지표:** 검색 청크 Hit@5·MRR@5, 반환 문맥의 Hit@5·고정 근거 Recall@5, 예산 적용 후 같은 지표, 전달 토큰량과 지연시간.
Recall은 후보별 청크 개수가 아닌 **동일한 정답 인용문 ID 목록**을 분모로 써 비교합니다. 의미상 모든 관련 근거를 판정한 완전한 relevance는 아닙니다.
단일 질문은 정답 인용문이 1개라 Hit와 Recall이 같고, 복합 질문은 2개 중 몇 개를 찾았는지 구분됩니다.

**제외:** 기업·연도 식별, 구조화 SQL 검색, reranker, LLM 답변·할루시네이션 평가는 없습니다. 부모는 E5에 임베딩하지 않습니다.
`skn25`에서 위에서 아래로 실행·저장하세요. Codex는 저장된 결과를 보고 해석하고 그때 HTML 보고서를 작성합니다.
OpenDART만 호출하며 OpenAI/LLM 과금과 Qwen GPU 설정 변경은 없습니다. 기존 notebook 전체·출력은 실행하지 않습니다.

## 1. 환경·검증된 함수와 데이터 계약 재사용

28번에서 원문 추출·E5·Kiwi·BM25 함수 5개, 29번에서 RRF 함수·고정 데이터·공식 보고서 선택 코드를 재사용합니다.
AST 해시를 실행 전에 검사합니다. 함수·데이터 계약이 바뀌면 중단하며, 기존 notebook의 출력 저장만 바뀐 것은 허용합니다.
28·29번을 먼저 실행할 필요는 없습니다. 이것은 실험용 notebook 의존성이며 최종 운영 코드의 방식이 아닙니다.

In [2]:
from io import BytesIO
from pathlib import Path
from time import perf_counter
from datetime import datetime, timezone
import ast, hashlib, importlib.metadata, json, os, re, sqlite3, sys, zipfile
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup, Comment, NavigableString, Tag
from dotenv import load_dotenv
from IPython.display import display
from kiwipiepy import Kiwi
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter, HierarchicalNodeParser, get_leaf_nodes
from transformers import AutoModel, AutoTokenizer
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn.functional as F

if Path(sys.prefix).name != 'skn25':
    raise RuntimeError('conda skn25 커널에서만 실행하세요.')
if importlib.metadata.version('llama-index-core') != '0.14.25':
    raise RuntimeError('검증한 llama-index-core 0.14.25와 다릅니다. 계층 파서 재검토가 필요합니다.')
PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                    if (p / 'src/dart/client.py').is_file())
load_dotenv(PROJECT_ROOT / '.env')
API_KEY = os.getenv('OPENDART_API_KEY')
if not API_KEY:
    raise RuntimeError('.env의 유효한 OPENDART_API_KEY가 필요합니다.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from dart.client import _get_bytes, CORP_CODE_URL

nb28 = json.loads((PROJECT_ROOT / 'notebooks/v1/28_hybrid_retrieval_paired_comparison.ipynb').read_text())
nb29 = json.loads((PROJECT_ROOT / 'notebooks/v1/29_rrf_weight_comparison.ipynb').read_text())
core_names = {'download_xml', 'extract_b_text', 'morph_terms', 'encode_texts', 'bm25_order'}
helpers = [n for c in nb28['cells'] if c['cell_type'] == 'code'
           for n in ast.parse(''.join(c['source'])).body
           if isinstance(n, ast.FunctionDef) and n.name in core_names]
helpers += [n for c in nb29['cells'] if c['cell_type'] == 'code'
            for n in ast.parse(''.join(c['source'])).body
            if isinstance(n, ast.FunctionDef) and n.name == 'weighted_rrf']
data_nodes = [n for c in nb29['cells'] if c['cell_type'] == 'code'
              for n in ast.parse(''.join(c['source'])).body
              if isinstance(n, ast.Assign) and any(isinstance(t, ast.Name) and t.id in {'REPORTS','CASES'} for t in n.targets)]
source_gate = ast.parse(''.join(next(c for c in nb29['cells'] if c.get('id') == 'v29-06')['source']))
reused_ast = ast.Module(body=helpers + data_nodes + source_gate.body, type_ignores=[])
reuse_sha256 = hashlib.sha256(ast.dump(reused_ast).encode()).hexdigest()
if len(helpers) != 6 or len(data_nodes) != 2 or reuse_sha256 != '0c51db7110a13828d60f8a92999c33ef19661bc0dc5d7fce16c68862082c3eab':
    raise RuntimeError('재사용 함수·데이터 계약이 검토된 버전과 다릅니다. 재검토 전 실행하지 않습니다.')
MODEL_ID, MODEL_REVISION = 'intfloat/multilingual-e5-small', '5697a65b0a002a92fe8c4fc9d495303ffff9c7d2'
CANDIDATE_DEPTH, RRF_K, EVAL_K = 20, 60, 5
VECTOR_WEIGHT, CHILD_SIZE, PARENT_SIZE, OVERLAP, CONTEXT_BUDGET = 0.5, 400, 1200, 48, 2400
METHODS = ('title_boundary', 'parent_child', 'heading_parent_child')
compact = lambda text: re.sub(r'[\s|]+', '', text)
exec(compile(ast.Module(body=helpers + data_nodes, type_ignores=[]), '<approved-functions-data>', 'exec'), globals())
print('skn25 | 검색 5:5 | E5 CPU | 부모 임베딩·LLM 호출 없음 | 기존 notebook 출력 실행 없음')


skn25 | 검색 5:5 | E5 CPU | 부모 임베딩·LLM 호출 없음 | 기존 notebook 출력 실행 없음


## 2. 공통 질문·정답 근거 목록 고정

복합 질문은 기존 근거 두 개를 함께 요구합니다. 예를 들어 화장품·생활용품·음료의 사업 구분과 계절적 매출 보완을 모두 확인해야 합니다.
정답 근거가 검색 질문이나 임베딩 입력에 들어가지 않으며, 평가할 때만 사용합니다. 새 회사·문서·독립 정답을 확보한 것은 아닙니다.

In [3]:
reference_quotes = {case['id']: case['quote'] for case in CASES}
qcases = [{**{k: case[k] for k in ('id','company','question','topic')},
           'dataset_group': 'single_25', 'reference_ids': [case['id']]} for case in CASES]
qcases += [
    {'id':'M_S','company':'삼성전자','topic':'복합','dataset_group':'multi_5','reference_ids':['S04','S05'],
     'question':'삼성전자의 반도체 사업과 하만 전장 사업이 각각 어떤 제품을 만드는지 비교해줘.'},
    {'id':'M_R','company':'리노공업','topic':'복합','dataset_group':'multi_5','reference_ids':['R02','R05'],
     'question':'리노공업의 해외 거래처 관리 방식과 원자재·에너지 관련 위험을 함께 정리해줘.'},
    {'id':'M_N','company':'NAVER','topic':'복합','dataset_group':'multi_5','reference_ids':['N01','N02'],
     'question':'네이버 서치플랫폼과 쇼핑 사업의 수익원을 비교하고 쇼핑의 광고 외 수익도 설명해줘.'},
    {'id':'M_L','company':'LG생활건강','topic':'복합','dataset_group':'multi_5','reference_ids':['L01','L04'],
     'question':'LG생활건강의 사업 구분과 화장품·음료 사업의 계절적 매출 보완 관계를 함께 설명해줘.'},
    {'id':'M_H','company':'HDC','topic':'복합','dataset_group':'multi_5','reference_ids':['H01','H04'],
     'question':'HDC 지주회사 자체의 수익원과 아이파크몰 입점 사업자를 통한 임대수익 구조를 구분해줘.'},
]
assert len(qcases) == 30 and len({case['id'] for case in qcases}) == 30
assert all(ref in reference_quotes for case in qcases for ref in case['reference_ids'])
case_company = {case['id']: case['company'] for case in CASES}
assert all(case_company[ref] == case['company'] for case in qcases for ref in case['reference_ids'])
evaluation_sha256 = hashlib.sha256(json.dumps({'questions':qcases,'references':reference_quotes}, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
display(pd.DataFrame(qcases)[['id','company','dataset_group','question','reference_ids']])
print('고정 질문·근거 SHA256:', evaluation_sha256)


,id,company,dataset_group,question,reference_ids
0,S01,삼성전자,single_25,삼성전자가 주식을 일반에 공개한 시점은 언제인가?,[S01]
1,S02,삼성전자,single_25,삼성전자공업에서 지금의 회사명으로 바뀐 날짜는?,[S02]
2,S03,삼성전자,single_25,삼성전자 본사의 소재지는 어디인가?,[S03]
3,S04,삼성전자,single_25,삼성전자 반도체 사업 부문은 어떤 제품을 주로 만드는가?,[S04]
4,S05,삼성전자,single_25,삼성전자 전장 사업에 해당하는 하만의 주요 제품은?,[S05]
5,R01,리노공업,single_25,리노공업은 납품 단가를 정할 때 고객 주문에서 무엇을 살피는가?,[R01]
6,R02,리노공업,single_25,리노공업의 해외 거래처 관리 방식은 무엇인가?,[R02]
7,R03,리노공업,single_25,리노공업의 공정가치로 측정되는 금융상품 관련 공시 근거를 찾아줘.,[R03]
8,R04,리노공업,single_25,리노공업 퇴직급여 표에서 확정급여채무와 함께 어떤 적립자산 항목을 확인할 수 있나?,[R04]
9,R05,리노공업,single_25,리노공업이 시장 환경에서 경계하는 원자재·에너지 관련 요인은?,[R05]


고정 질문·근거 SHA256: a8bbc6202b3b0956c7519660a6e760eb1dab9be1d808118bdba1e7557c3c6ebe


## 3. 동일한 공식 보고서 선택과 텍스트 추출

29번의 기업코드·제출일·사업연도·보고서명·접수번호·표지 검사 코드를 그대로 사용합니다.
같은 다운로드의 XML도 C의 제목 계층 확인을 위해 메모리에 보존합니다. 중복 API 호출은 없습니다. 다운로드는 이번 커널 메모리에서 처리하며 원본·인덱스의 운영 저장 정책을 새로 구현하지 않습니다. 키·요청 URL은 출력하지 않습니다.

In [4]:
# 검증된 원문 gate는 유지하고, 같은 다운로드의 XML을 계층 확인용으로 메모리에 보존합니다.
source_xml_by_receipt = {}
gate_download_xml = download_xml
def keep_xml_for_hierarchy(receipt, filename):
    data = gate_download_xml(receipt, filename)
    source_xml_by_receipt[receipt] = data
    return data

download_xml = keep_xml_for_hierarchy
try:
    exec(compile(source_gate, '<29-approved-source-gate>', 'exec'), globals())
finally:
    download_xml = gate_download_xml
source_xml = {row['company']: source_xml_by_receipt[row['receipt']] for row in source_rows}
assert set(source_texts) == set(source_xml) == set(REPORTS)
del source_xml_by_receipt


삼성전자 원문 준비 시작 | 접수번호 20250311001085
삼성전자 준비 완료 | 44,847자
리노공업 원문 준비 시작 | 접수번호 20250318000229
리노공업 준비 완료 | 159,669자
NAVER 원문 준비 시작 | 접수번호 20250318000645
NAVER 준비 완료 | 639,240자
LG생활건강 원문 준비 시작 | 접수번호 20250317000790
LG생활건강 준비 완료 | 426,665자
HDC 원문 준비 시작 | 접수번호 20250318001151
HDC 준비 완료 | 62,037자


,company,corp_code,business_year,receipt,xml_MB,text_chars,text_utf8_MB,source_sha256,download_extract_seconds,source_url
0,삼성전자,00126380,2024,20250311001085,6.006890,44847,0.079431,b0682fd46de50ecc41f074bb43cb58cf574dd175034371...,0.386865,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
1,리노공업,00369657,2024,20250318000229,1.486798,159669,0.298023,a23af2dbcc2f4de4be59c5925ed45fceae84c23f4e78ff...,0.709838,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
2,NAVER,00266961,2024,20250318000645,6.900676,639240,1.079369,b1478f612c3ff07373d14fe9c7638936c77fcea2130aa4...,4.080670,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
3,LG생활건강,00356370,2024,20250317000790,4.573826,426665,0.764530,9ccc7d18e1ba66ce3982f41899d22d1dcb7ae8dfaf784f...,1.987121,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
4,HDC,00164636,2024,20250318001151,6.137928,62037,0.108997,53b41f2833b68f06bfea13a07abed1a7cdb3caf53c44e6...,0.440156,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...


## 4. A/B: 제목 경계 분할과 토큰 크기 기반 부모·자식

E5 기준으로 토큰 수를 세되, 부모는 E5 입력 한도보다 커도 검색 모델에 넣지 않습니다.
자식/작은 청크만 `passage:`를 포함해 512-token 이하인지 검사합니다. 두 후보 모두 같은 제목 섹션을 입력으로 사용합니다.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
def count_tokens(text):
    return len(tokenizer.encode(text, add_special_tokens=False, verbose=False))

def make_candidate(text, method, xml_bytes=None):
    if method == 'heading_parent_child':
        return make_heading_candidate(text, xml_bytes)
    sections = [s for s in re.split(r'(?=^## )', text, flags=re.MULTILINE) if s.strip()]
    docs = [Document(text=s) for s in sections]
    small = SentenceSplitter(chunk_size=CHILD_SIZE, chunk_overlap=OVERLAP, tokenizer=lambda s: tokenizer.encode(s, add_special_tokens=False, verbose=False))
    if method == 'title_boundary':
        leaves = small.get_nodes_from_documents(docs)
        units = {n.node_id: n.text.strip() for n in leaves}
        payload_ids = [n.node_id for n in leaves]
    elif method == 'parent_child':
        parent = SentenceSplitter(chunk_size=PARENT_SIZE, chunk_overlap=OVERLAP, tokenizer=lambda s: tokenizer.encode(s, add_special_tokens=False, verbose=False))
        hierarchy = HierarchicalNodeParser.from_defaults(node_parser_ids=['parent','child'],
                    node_parser_map={'parent':parent,'child':small})
        nodes = hierarchy.get_nodes_from_documents(docs)
        leaves = get_leaf_nodes(nodes)
        parents = {n.node_id:n for n in nodes if n.child_nodes}
        units = {node_id:n.text.strip() for node_id,n in parents.items()}
        payload_ids = []
        for leaf in leaves:
            if not leaf.parent_node or leaf.parent_node.node_id not in parents:
                raise RuntimeError('자식에 연결된 부모를 찾지 못했습니다.')
            parent_id = leaf.parent_node.node_id
            if not any(n.node_id == leaf.node_id for n in parents[parent_id].child_nodes):
                raise RuntimeError('부모→자식 역방향 연결 오류')
            if compact(leaf.text.strip()) not in compact(units[parent_id]):
                raise RuntimeError('자식 텍스트가 연결된 부모 문맥에 없습니다.')
            payload_ids.append(parent_id)
    else:
        raise ValueError('알 수 없는 청킹 후보')
    texts = [n.text.strip() for n in leaves]
    if not texts or not all(texts):
        raise RuntimeError('빈 검색 청크가 있습니다.')
    lengths = [len(tokenizer('passage: ' + t, truncation=False, verbose=False)['input_ids']) for t in texts]
    if max(lengths) > 512:
        raise RuntimeError('E5에 넣을 작은 청크가 512-token을 넘습니다. 잘라내지 않고 중단합니다.')
    return texts, payload_ids, units, max(lengths)


## 4-1. C: 원문 제목 수준으로 계층 만들기

같은 XML 파서·같은 추출 텍스트를 사용합니다. SECTION 중첩이 제목 수준을 정하며 SUBTITLE은 해당 SECTION의 TITLE 아래에 둡니다. 연속 제목 조각은 같은 제목으로 묶습니다. 실제 연결을 찾지 못하면 중단하고 크기 기반 방식으로 대체하지 않습니다. 본문의 번호·굵은 글씨를 추가 제목으로 추측하는 것은 제외합니다.


In [6]:
# extract_b_text의 동일한 XML 보정·파서 부분만 재사용합니다. 추출 정책을 C만 바꾸지 않습니다.
extract_def = next(node for node in helpers if node.name == 'extract_b_text')
parse_prefix = []
for statement in extract_def.body:
    parse_prefix.append(statement)
    if isinstance(statement, ast.Assign) and any(isinstance(t, ast.Name) and t.id == 'soup' for t in statement.targets):
        break
else:
    raise RuntimeError('검증된 추출 함수의 XML 파서 부분을 찾지 못했습니다.')
parse_def = ast.FunctionDef(name='parse_source_tree', args=extract_def.args,
    body=parse_prefix + [ast.Return(value=ast.Name(id='soup', ctx=ast.Load()))], decorator_list=[])
exec(compile(ast.fix_missing_locations(ast.Module(body=[parse_def], type_ignores=[])), '<same-xml-parser>', 'exec'), globals())

def heading_tree(xml_bytes, expected_text):
    soup = parse_source_tree(xml_bytes)
    # 원래 B formatter는 제목의 각 텍스트 조각마다 ## 표시를 남깁니다.
    events = []
    for node in soup.descendants:
        if not isinstance(node, NavigableString) or isinstance(node, Comment) or not str(node).strip():
            continue
        heading = next((p for p in node.parents if p.name and p.name.upper() in {'TITLE','SUBTITLE'}), None)
        if heading is not None:
            section_path = tuple(id(p) for p in reversed(list(heading.parents))
                                 if p.name and re.fullmatch(r'SECTION-\d+', p.name.upper()))
            level = max(1, len(section_path)) + int(heading.name.upper() == 'SUBTITLE')
            events.append({'tag_id':id(heading), 'level':level, 'section_path':section_path,
                           'title':heading.get_text(' ', strip=True), 'tag':heading.name.upper()})
    blocks = [b for b in re.split(r'(?=^## )', expected_text, flags=re.MULTILINE) if b.strip()]
    preamble = blocks.pop(0) if blocks and not blocks[0].startswith('## ') else ''
    if len(blocks) != len(events) or not events:
        raise RuntimeError('XML 제목과 추출 텍스트 경계가 일치하지 않습니다. 계층을 임의 추측하지 않습니다.')
    records = []
    for event, block in zip(events, blocks):
        if records and records[-1]['tag_id'] == event['tag_id']:
            records[-1]['own_text'] += block
        else:
            records.append({**event, 'id':f'heading:{len(records)}', 'own_text':block,
                            'parent_id':None, 'end':None, 'children':[]})
    if compact(preamble + ''.join(r['own_text'] for r in records)) != compact(expected_text):
        raise RuntimeError('계층 분할 과정에서 원문 텍스트가 달라졌습니다.')
    stack = []
    for i, record in enumerate(records):
        while stack:
            parent = records[stack[-1]]
            path, parent_path = record['section_path'], parent['section_path']
            same_branch = path[:len(parent_path)] == parent_path and (bool(parent_path) or not path)
            if record['level'] > parent['level'] and same_branch:
                break
            records[stack.pop()]['end'] = i
        if stack:
            parent = records[stack[-1]]
            record['parent_id'] = parent['id']
            parent['children'].append(record['id'])
        stack.append(i)
    for i in stack:
        records[i]['end'] = len(records)
    for i, record in enumerate(records):
        record['context'] = ''.join(r['own_text'] for r in records[i:record['end']]).strip()
    if not any(r['parent_id'] for r in records):
        raise RuntimeError('실제 제목 계층의 부모·자식 관계가 없습니다. 토큰 방식으로 조용히 대체하지 않습니다.')
    return records, preamble


## 4-2. C: 제목 구간 검색과 실제 부모 반환

E5 한도 안에 있는 제목 구간은 그대로 검색합니다. 한도를 넘는 본문만 안전하게 재분할하며 원래 부모·자식 제목 관계는 유지합니다. 부모에는 고정 1,200-token 제한을 두지 않습니다. 원래 제목 부모가 너무 크면 공통 2,400-token 예산에서는 반환하지 못할 수 있으므로, 무제한 결과 및 예산 불일치를 함께 확인해야 합니다.


In [7]:
def make_heading_candidate(text, xml_bytes):
    records, preamble = heading_tree(xml_bytes, text)
    all_units = {r['id']:r['context'] for r in records}
    texts, payload_ids = [], []
    overflow_count = 0
    safeguard = SentenceSplitter(chunk_size=CHILD_SIZE, chunk_overlap=OVERLAP,
        tokenizer=lambda s: tokenizer.encode(s, add_special_tokens=False, verbose=False))
    for record in records:
        own = record['own_text'].strip()
        # 내용 없이 하위 제목만 묶는 제목은 부모로만 보존하고 검색 leaf에는 넣지 않습니다.
        if record['children'] and not re.sub(r'^## .*$', '', own, flags=re.MULTILINE).strip():
            continue
        # 자연 제목 leaf(예: 국내 전용)는 실제 상위 제목(연회비)을 반환합니다.
        # 하위 제목이 있는 내부 노드의 직접 본문은 해당 노드 자신을 반환합니다.
        payload = record['id'] if record['children'] else (record['parent_id'] or record['id'])
        if len(tokenizer('passage: ' + own, truncation=False, verbose=False)['input_ids']) <= 512:
            pieces = [own]
        else:
            overflow_count += 1
            pieces = [n.text.strip() for n in safeguard.get_nodes_from_documents([Document(text=own)])]
        for piece in pieces:
            if compact(piece) not in compact(all_units[payload]):
                raise RuntimeError('제목 자식 텍스트가 실제 부모 문맥에 없습니다.')
            texts.append(piece); payload_ids.append(payload)
    if preamble.strip():
        all_units['preamble'] = preamble.strip()
        pieces = [n.text.strip() for n in safeguard.get_nodes_from_documents([Document(text=preamble)])]
        texts.extend(pieces); payload_ids.extend(['preamble'] * len(pieces))
    if not texts or not all(texts):
        raise RuntimeError('제목 계층 검색 단위가 비어 있습니다.')
    lengths = [len(tokenizer('passage: ' + t, truncation=False, verbose=False)['input_ids']) for t in texts]
    if max(lengths) > 512:
        raise RuntimeError('제목 검색 조각이 E5 한도를 넘습니다. 임의 truncate 없이 중단합니다.')
    heading_diagnostics.append({'company':current_company, 'heading_nodes':len(records),
        'structural_edges':sum(r['parent_id'] is not None for r in records),
        'oversized_heading_bodies':overflow_count, 'preamble_search_pieces':len(pieces) if preamble.strip() else 0})
    heading_examples.extend({'company':current_company, 'id':r['id'], 'level':r['level'], 'title':r['title'],
        'parent_id':r['parent_id'], 'context_e5_tokens':count_tokens(r['context'])} for r in records)
    units = {i:all_units[i] for i in set(payload_ids)}
    return texts, payload_ids, units, max(lengths)

# 계층은 token 숫자가 아니라 원문 SECTION/TITLE로 결정되는지 확인합니다.
probe_xml = b'<DOCUMENT><SECTION-1><TITLE>Fees</TITLE><SECTION-2><TITLE>Domestic</TITLE><P>DOMESTIC_FEE</P></SECTION-2><SECTION-2><TITLE>Overseas</TITLE><P>OVERSEAS_FEE</P></SECTION-2></SECTION-1><SECTION-1><TITLE>Benefits</TITLE><P>BENEFIT</P></SECTION-1></DOCUMENT>'
probe_tree, _ = heading_tree(probe_xml, extract_b_text(probe_xml))
assert [r['level'] for r in probe_tree] == [1,2,2,1]
assert probe_tree[1]['parent_id'] == probe_tree[2]['parent_id'] == probe_tree[0]['id']
assert probe_tree[3]['parent_id'] is None
assert 'DOMESTIC_FEE' in probe_tree[0]['context'] and 'OVERSEAS_FEE' in probe_tree[0]['context']
assert 'BENEFIT' not in probe_tree[0]['context']
# 반환 정책도 검사: 실제 제목 leaf→상위 제목, 내부 노드 직접본문→자신, 독립 제목→자신.
heading_diagnostics, heading_examples, current_company = [], [], '__probe__'
policy_xml = probe_xml.replace(b'<TITLE>Fees</TITLE>', b'<TITLE>Fees</TITLE><P>GENERAL_FEE</P>')
policy_text = extract_b_text(policy_xml)
policy_leaves, policy_ids, policy_units, _ = make_heading_candidate(policy_text, policy_xml)
general_idx = next(i for i, t in enumerate(policy_leaves) if 'GENERAL_FEE' in t)
domestic_idx = next(i for i, t in enumerate(policy_leaves) if 'DOMESTIC_FEE' in t)
overseas_idx = next(i for i, t in enumerate(policy_leaves) if 'OVERSEAS_FEE' in t)
benefit_idx = next(i for i, t in enumerate(policy_leaves) if 'BENEFIT' in t)
assert policy_ids[general_idx] == policy_ids[domestic_idx] == policy_ids[overseas_idx]
assert policy_ids[benefit_idx] != policy_ids[domestic_idx]
assert 'OVERSEAS_FEE' in policy_units[policy_ids[domestic_idx]]
assert 'BENEFIT' not in policy_units[policy_ids[domestic_idx]]
del heading_diagnostics, heading_examples, current_company
print('원문 제목 계층·형제 구분·부모 본문 보존·실제 반환 정책 검사 완료')


원문 제목 계층·형제 구분·부모 본문 보존·실제 반환 정책 검사 완료


## 5. 분할 결과와 근거 보존 검사

원문·A 청크·B/C 부모에 정답 인용문이 없으면 중단합니다. B 자식에서 인용문이 경계에 걸려 온전히 남지 않은 경우는 기록하되,
부모에 보존돼 있으면 제외하지 않습니다. 그런 경우야말로 부모 확장이 개선할 수 있는지를 볼 대상입니다.

In [8]:
texts_by_key, payload_ids_by_key, context_units_by_key = {}, {}, {}
chunk_rows, audit_rows = [], []
heading_diagnostics, heading_examples = [], []
for method in METHODS:
    for company, text in source_texts.items():
        key = (method, company)
        started = perf_counter()
        current_company = company
        texts, payload_ids, units, max_tokens = make_candidate(text, method, source_xml[company])
        texts_by_key[key], payload_ids_by_key[key], context_units_by_key[key] = texts, payload_ids, units
        chunk_rows.append({'method':method,'company':company,'search_chunks':len(texts),
                           'return_units':len(units),'max_e5_tokens':max_tokens,
                           'chunking_seconds':perf_counter()-started,
                           'search_text_MiB':sum(len(t.encode()) for t in texts)/1024**2,
                           'context_text_MiB':sum(len(t.encode()) for t in units.values())/1024**2})
        print(f'{method} | {company} | 검색 청크 {len(texts)} | 반환 단위 {len(units)}', flush=True)
        for case in [c for c in CASES if c['company'] == company]:
            needle = compact(case['quote'])
            audit_rows.append({'method':method,'id':case['id'],'company':company,
                'source_exists':needle in compact(text),
                'search_gold_chunks':sum(needle in compact(t) for t in texts),
                'return_gold_units':sum(needle in compact(t) for t in units.values())})
chunk_info, gold_audit = pd.DataFrame(chunk_rows), pd.DataFrame(audit_rows)
display(chunk_info); display(gold_audit)
display(pd.DataFrame(heading_diagnostics))
with pd.option_context('display.max_colwidth', 90):
    display(pd.DataFrame(heading_examples).groupby('company', sort=False).head(15))
bad = gold_audit[(~gold_audit['source_exists']) | (gold_audit['return_gold_units'] == 0)
                 | ((gold_audit['method'] == 'title_boundary') & (gold_audit['search_gold_chunks'] == 0))]
if not bad.empty:
    raise RuntimeError('원문·반환 문맥의 정답 근거 보존 실패. 문항을 빼지 말고 추출·분할을 확인하세요.')


title_boundary | 삼성전자 | 검색 청크 82 | 반환 단위 82
title_boundary | 리노공업 | 검색 청크 333 | 반환 단위 333
title_boundary | NAVER | 검색 청크 1168 | 반환 단위 1168
title_boundary | LG생활건강 | 검색 청크 824 | 반환 단위 824
title_boundary | HDC | 검색 청크 123 | 반환 단위 123
parent_child | 삼성전자 | 검색 청크 91 | 반환 단위 32
parent_child | 리노공업 | 검색 청크 349 | 반환 단위 146
parent_child | NAVER | 검색 청크 1293 | 반환 단위 418
parent_child | LG생활건강 | 검색 청크 902 | 반환 단위 325
parent_child | HDC | 검색 청크 134 | 반환 단위 46
heading_parent_child | 삼성전자 | 검색 청크 79 | 반환 단위 5
heading_parent_child | 리노공업 | 검색 청크 316 | 반환 단위 18
heading_parent_child | NAVER | 검색 청크 1155 | 반환 단위 17
heading_parent_child | LG생활건강 | 검색 청크 805 | 반환 단위 16
heading_parent_child | HDC | 검색 청크 120 | 반환 단위 5


,method,company,search_chunks,return_units,max_e5_tokens,chunking_seconds,search_text_MiB,context_text_MiB
0,title_boundary,삼성전자,82,82,403,0.252669,0.082886,0.082886
1,title_boundary,리노공업,333,333,404,0.536452,0.309031,0.309031
2,title_boundary,NAVER,1168,1168,404,2.871409,1.146790,1.146790
3,title_boundary,LG생활건강,824,824,404,1.725481,0.811188,0.811188
4,title_boundary,HDC,123,123,404,0.240656,0.115170,0.115170
5,parent_child,삼성전자,91,32,403,0.223959,0.084018,0.080358
6,parent_child,리노공업,349,146,404,0.797465,0.309011,0.299274
7,parent_child,NAVER,1293,418,404,4.325515,1.153458,1.096161
8,parent_child,LG생활건강,902,325,404,2.824909,0.815345,0.773008
9,parent_child,HDC,134,46,399,0.389932,0.115545,0.110558


,method,id,company,source_exists,search_gold_chunks,return_gold_units
0,title_boundary,S01,삼성전자,True,1,1
1,title_boundary,S02,삼성전자,True,1,1
2,title_boundary,S03,삼성전자,True,4,4
3,title_boundary,S04,삼성전자,True,2,2
4,title_boundary,S05,삼성전자,True,1,1
...,...,...,...,...,...,...
70,heading_parent_child,H01,HDC,True,1,1
71,heading_parent_child,H02,HDC,True,2,1
72,heading_parent_child,H03,HDC,True,1,1
73,heading_parent_child,H04,HDC,True,1,1


,company,heading_nodes,structural_edges,oversized_heading_bodies,preamble_search_pieces
0,삼성전자,16,12,10,1
1,리노공업,98,82,41,1
2,NAVER,135,131,94,1
3,LG생활건강,154,149,93,1
4,HDC,14,10,9,1


,company,id,level,title,parent_id,context_e5_tokens
0,삼성전자,heading:0,1,목 차,NaN,3
1,삼성전자,heading:1,1,【 대표이사 등의 확인 】,NaN,18
2,삼성전자,heading:2,1,I. 회사의 개요,NaN,9732
3,삼성전자,heading:3,2,1. 회사의 개요,heading:2,2750
4,삼성전자,heading:4,2,2. 회사의 연혁,heading:2,2082
...,...,...,...,...,...,...
412,HDC,heading:9,2,1. 사업의 개요,heading:8,1086
413,HDC,heading:10,2,2. 주요 제품 및 서비스,heading:8,2602
414,HDC,heading:11,2,3. 원재료 및 생산설비,heading:8,10214
415,HDC,heading:12,2,4. 매출 및 수주상황,heading:8,9108


## 6. 같은 Kiwi·BM25 검색 조건으로 세 인덱스 만들기

BM25는 A의 작은 청크/B의 토큰 자식/C의 제목 구간·안전 분할 조각만 인덱싱합니다. 부모 텍스트를 BM25에 추가하지 않습니다.
회사와 후보별 메모리 DB를 분리해 다른 기업·다른 후보의 근거가 섞이지 않게 합니다.

In [9]:
kiwi = Kiwi()
for connection in globals().get('bm25_connections', {}).values():
    connection.close()
bm25_connections, bm25_rows = {}, []
for key, texts in texts_by_key.items():
    started = perf_counter()
    connection = sqlite3.connect(':memory:')
    connection.execute('CREATE VIRTUAL TABLE passage USING fts5(text)')
    terms = [' '.join(morph_terms(text)) for text in texts]
    connection.executemany('INSERT INTO passage(rowid, text) VALUES (?,?)', enumerate(terms, 1))
    connection.commit()
    bm25_connections[key] = connection
    pages = connection.execute('PRAGMA page_count').fetchone()[0]
    page_size = connection.execute('PRAGMA page_size').fetchone()[0]
    bm25_rows.append({'method':key[0],'company':key[1],'bm25_build_seconds':perf_counter()-started,
                     'sqlite_MiB':pages*page_size/1024**2,'empty_bm25_chunks':sum(not t for t in terms)})
display(pd.DataFrame(bm25_rows))


,method,company,bm25_build_seconds,sqlite_MiB,empty_bm25_chunks
0,title_boundary,삼성전자,1.781508,0.140625,0
1,title_boundary,리노공업,1.487323,0.421875,0
2,title_boundary,NAVER,4.737773,1.476562,1
3,title_boundary,LG생활건강,3.815310,1.031250,0
4,title_boundary,HDC,0.516250,0.179688,0
5,parent_child,삼성전자,0.397734,0.140625,0
6,parent_child,리노공업,1.609407,0.417969,0
7,parent_child,NAVER,4.833820,1.476562,2
8,parent_child,LG생활건강,4.189770,1.027344,0
9,parent_child,HDC,0.549753,0.183594,0


## 7. E5 CPU 임베딩 — 부모는 임베딩하지 않음

E5를 한 번 로드합니다. 세 후보에 완전히 같은 청크 텍스트가 있으면 그 벡터만 재사용합니다.
문서 처리 진행률은 64개마다 표시합니다. 임베딩 준비 비용은 재사용 개수를 함께 해석해야 하며, 후보별 독립 cold 시간 비교가 아닙니다.

In [10]:
started = perf_counter()
print('E5 CPU 모델 로드 시작', flush=True)
model = AutoModel.from_pretrained(MODEL_ID, revision=MODEL_REVISION, use_safetensors=True).cpu().eval()
model_load_seconds = perf_counter()-started
parameter_MiB = sum(p.numel()*p.element_size() for p in model.parameters())/1024**2
embedding_cache, dense_index, e5_rows = {}, {}, []
for key, texts in texts_by_key.items():
    started = perf_counter()
    missing = [t for t in dict.fromkeys(texts) if t not in embedding_cache]
    print(f'{key} | {len(texts)}청크 | 새 임베딩 {len(missing)}개', flush=True)
    for start in range(0, len(missing), 64):
        batch = missing[start:start+64]
        vectors = encode_texts(batch, 'passage')
        embedding_cache.update(zip(batch, vectors))
        print(f'  {min(start+64,len(missing))}/{len(missing)} 새 청크 완료', flush=True)
    dense_index[key] = np.stack([embedding_cache[t] for t in texts])
    e5_rows.append({'method':key[0],'company':key[1],'new_embedding_texts':len(missing),
                   'reused_records':len(texts)-len(missing),'e5_build_seconds':perf_counter()-started,
                   'vectors_MiB':dense_index[key].nbytes/1024**2})
display(pd.DataFrame(e5_rows))
print(f'E5 CPU 로드 {model_load_seconds:.2f}초 | 파라미터 메모리 {parameter_MiB:.1f} MiB')


E5 CPU 모델 로드 시작
('title_boundary', '삼성전자') | 82청크 | 새 임베딩 82개
  64/82 새 청크 완료
  82/82 새 청크 완료
('title_boundary', '리노공업') | 333청크 | 새 임베딩 331개
  64/331 새 청크 완료
  128/331 새 청크 완료
  192/331 새 청크 완료
  256/331 새 청크 완료
  320/331 새 청크 완료
  331/331 새 청크 완료
('title_boundary', 'NAVER') | 1168청크 | 새 임베딩 1156개
  64/1156 새 청크 완료
  128/1156 새 청크 완료
  192/1156 새 청크 완료
  256/1156 새 청크 완료
  320/1156 새 청크 완료
  384/1156 새 청크 완료
  448/1156 새 청크 완료
  512/1156 새 청크 완료
  576/1156 새 청크 완료
  640/1156 새 청크 완료
  704/1156 새 청크 완료
  768/1156 새 청크 완료
  832/1156 새 청크 완료
  896/1156 새 청크 완료
  960/1156 새 청크 완료
  1024/1156 새 청크 완료
  1088/1156 새 청크 완료
  1152/1156 새 청크 완료
  1156/1156 새 청크 완료
('title_boundary', 'LG생활건강') | 824청크 | 새 임베딩 809개
  64/809 새 청크 완료
  128/809 새 청크 완료
  192/809 새 청크 완료
  256/809 새 청크 완료
  320/809 새 청크 완료
  384/809 새 청크 완료
  448/809 새 청크 완료
  512/809 새 청크 완료
  576/809 새 청크 완료
  640/809 새 청크 완료
  704/809 새 청크 완료
  768/809 새 청크 완료
  809/809 새 청크 완료
('title_boundary', 'HDC') | 123청크 | 새 임베딩 120개
  64/1

,method,company,new_embedding_texts,reused_records,e5_build_seconds,vectors_MiB
0,title_boundary,삼성전자,82,0,3.212246,0.120117
1,title_boundary,리노공업,331,2,12.674248,0.487793
2,title_boundary,NAVER,1156,12,45.182544,1.710938
3,title_boundary,LG생활건강,809,15,31.531195,1.207031
4,title_boundary,HDC,120,3,4.540928,0.180176
5,parent_child,삼성전자,32,59,1.170317,0.133301
6,parent_child,리노공업,84,265,3.251469,0.511230
7,parent_child,NAVER,668,625,27.357229,1.894043
8,parent_child,LG생활건강,395,507,15.144419,1.321289
9,parent_child,HDC,73,61,2.845222,0.196289


E5 CPU 로드 0.22초 | 파라미터 메모리 448.8 MiB


## 8. 상위 5개에서 반환 문맥 만들기와 공통 예산

B/C는 검색된 자식의 부모를 순서대로 가져오고 동일 부모를 중복 제거합니다. A는 작은 청크 그대로입니다.
반환 문맥의 Recall은 정답 인용문 ID 목록 중 온전히 들어 있는 인용문 비율입니다. 이 점수는 LLM의 이해·답변 품질을 대신하지 않습니다.
예산은 E5 토크나이저의 텍스트 환산값이며 실제 운영 LLM의 입력 토큰이나 API 청구 토큰이 아닙니다.

In [11]:
def returned_contexts(key, order, k=EVAL_K):
    unique, seen = [], set()
    for idx in order[:k]:
        unit_id = payload_ids_by_key[key][int(idx)]
        if unit_id not in seen:
            unique.append(context_units_by_key[key][unit_id])
            seen.add(unit_id)
    return unique

def pack_contexts(units, budget=CONTEXT_BUDGET):
    accepted = []
    for unit in units:
        if count_tokens('\n\n'.join(accepted + [unit])) > budget:
            break
        accepted.append(unit)
    return accepted

def evidence_recall(units, reference_ids):
    if not reference_ids:
        raise ValueError('정답 근거 목록이 비어 있습니다.')
    return sum(any(compact(reference_quotes[ref]) in compact(unit) for unit in units)
               for ref in reference_ids)/len(reference_ids)

# 실제 반환 정책 회귀 검사: 부모 중복 제거, 더 많은 자식으로 보충하지 않음, 예산 초과/부분 문맥 금지.
probe_key = ('probe','company')
payload_ids_by_key[probe_key] = ['p','p','q']
context_units_by_key[probe_key] = {'p':'AAAA','q':'BBBB'}
assert returned_contexts(probe_key, [0,1,2], k=2) == ['AAAA']
assert returned_contexts(probe_key, [0,1,2], k=3) == ['AAAA','BBBB']
assert pack_contexts(['AAAA','BBBB'], budget=count_tokens('AAAA')) == ['AAAA']
assert pack_contexts(['AAAA'], budget=0) == []
assert evidence_recall([reference_quotes['S04']], ['S04','S05']) == 0.5
assert evidence_recall([reference_quotes['S04'],reference_quotes['S05']], ['S04','S05']) == 1.0
del payload_ids_by_key[probe_key], context_units_by_key[probe_key]
print('부모 연결 반환·중복 제거·공통 예산·고정 근거 Recall 검사 완료')


부모 연결 반환·중복 제거·공통 예산·고정 근거 Recall 검사 완료


## 9. 단일·복합 질문 A/B/C 실행

질문 임베딩은 세 후보가 공유합니다. 검색은 후보별 다른 청크 인덱스에서 수행합니다.
각 문항마다 **검색 청크 Hit / 부모 확장 뒤 근거 Recall / 공통 예산 뒤 Recall** 진행값을 출력합니다.
부모 확장 점수가 좋아도 제공 문맥이 훨씬 길어진 효과인지 확인해야 합니다.

In [12]:
rows, details = [], []
for number, case in enumerate(qcases, 1):
    print(f"[{number}/{len(qcases)}] {case['id']} {case['company']} 검색 시작", flush=True)
    started = perf_counter()
    query_vector = encode_texts([case['question']], 'query')[0]
    query_embed_ms = (perf_counter()-started)*1000
    for method in METHODS:
        key, refs = (method,case['company']), case['reference_ids']
        started = perf_counter()
        lex_order = bm25_order(key, case['question'])
        dense_order = np.argsort(-(dense_index[key] @ query_vector), kind='stable')
        order = weighted_rrf(dense_order, lex_order, len(texts_by_key[key]), VECTOR_WEIGHT)
        search_ms = query_embed_ms + (perf_counter()-started)*1000
        first_rank = next((rank for rank, idx in enumerate(order, 1)
                          if evidence_recall([texts_by_key[key][int(idx)]], refs) > 0), None)
        leaf_units = [texts_by_key[key][int(idx)] for idx in order[:EVAL_K]]
        started = perf_counter()
        contexts = returned_contexts(key, order)
        bounded = pack_contexts(contexts)
        context_ms = (perf_counter()-started)*1000
        raw_recall, bounded_recall = evidence_recall(contexts, refs), evidence_recall(bounded, refs)
        rows.append({'id':case['id'],'company':case['company'],'dataset_group':case['dataset_group'],
                     'method':method,'references':len(refs),'first_leaf_rank':first_rank,
                     'leaf_hit_at_5':first_rank is not None and first_rank <= EVAL_K,
                     'leaf_mrr_at_5':1/first_rank if first_rank is not None and first_rank <= EVAL_K else 0.0,
                     'leaf_evidence_recall_at_5':evidence_recall(leaf_units, refs),
                     'context_hit_at_5':raw_recall > 0,'context_recall_at_5':raw_recall,
                     'budget_context_hit_at_5':bounded_recall > 0,'budget_context_recall_at_5':bounded_recall,
                     'raw_contexts':len(contexts),'packed_contexts':len(bounded),
                     'first_context_over_budget':bool(contexts) and count_tokens(contexts[0]) > CONTEXT_BUDGET,
                     'oversized_context_count':sum(count_tokens(u) > CONTEXT_BUDGET for u in contexts),
                     'raw_e5_tokens':count_tokens('\n\n'.join(contexts)),
                     'packed_e5_tokens':count_tokens('\n\n'.join(bounded)),
                     'search_ms':search_ms,'context_ms':context_ms})
        missing = [ref for ref in refs if evidence_recall(bounded,[ref]) == 0]
        details.append({'id':case['id'],'company':case['company'],'method':method,
                        'question':case['question'],'missing_reference_ids':missing,
                        'reference_quotes':[reference_quotes[r] for r in refs],
                        'top1_leaf':leaf_units[0][:240] if leaf_units else '',
                        'first_context_preview':contexts[0][:320] if contexts else ''})
        print(f"  {method}: leaf Hit={rows[-1]['leaf_hit_at_5']} | 문맥 Recall={raw_recall:.2f} | 예산 Recall={bounded_recall:.2f} | {rows[-1]['packed_e5_tokens']}tokens", flush=True)
results, detail_df = pd.DataFrame(rows), pd.DataFrame(details)
assert len(results) == len(qcases)*len(METHODS) and results.groupby('method').size().eq(len(qcases)).all()
assert results['packed_e5_tokens'].le(CONTEXT_BUDGET).all()
display(results)


[1/30] S01 삼성전자 검색 시작
  title_boundary: leaf Hit=True | 문맥 Recall=1.00 | 예산 Recall=1.00 | 1652tokens
  parent_child: leaf Hit=True | 문맥 Recall=1.00 | 예산 Recall=1.00 | 1376tokens
  heading_parent_child: leaf Hit=True | 문맥 Recall=1.00 | 예산 Recall=0.00 | 0tokens
[2/30] S02 삼성전자 검색 시작
  title_boundary: leaf Hit=True | 문맥 Recall=1.00 | 예산 Recall=1.00 | 1792tokens
  parent_child: leaf Hit=True | 문맥 Recall=1.00 | 예산 Recall=1.00 | 2114tokens
  heading_parent_child: leaf Hit=True | 문맥 Recall=1.00 | 예산 Recall=0.00 | 0tokens
[3/30] S03 삼성전자 검색 시작
  title_boundary: leaf Hit=True | 문맥 Recall=1.00 | 예산 Recall=1.00 | 1706tokens
  parent_child: leaf Hit=True | 문맥 Recall=1.00 | 예산 Recall=1.00 | 2052tokens
  heading_parent_child: leaf Hit=True | 문맥 Recall=1.00 | 예산 Recall=0.00 | 0tokens
[4/30] S04 삼성전자 검색 시작
  title_boundary: leaf Hit=True | 문맥 Recall=1.00 | 예산 Recall=1.00 | 1837tokens
  parent_child: leaf Hit=True | 문맥 Recall=1.00 | 예산 Recall=1.00 | 2052tokens
  heading_parent_child: leaf Hit=True | 문맥

,id,company,dataset_group,method,references,first_leaf_rank,leaf_hit_at_5,leaf_mrr_at_5,leaf_evidence_recall_at_5,context_hit_at_5,...,budget_context_hit_at_5,budget_context_recall_at_5,raw_contexts,packed_contexts,first_context_over_budget,oversized_context_count,raw_e5_tokens,packed_e5_tokens,search_ms,context_ms
0,S01,삼성전자,single_25,title_boundary,1,2.0,True,0.500000,1.0,True,...,True,1.0,5,5,False,0,1652,1652,20.252435,6.444414
1,S01,삼성전자,single_25,parent_child,1,1.0,True,1.000000,1.0,True,...,True,1.0,5,2,False,0,4640,1376,20.329283,5.542198
2,S01,삼성전자,single_25,heading_parent_child,1,2.0,True,0.500000,1.0,True,...,False,0.0,2,0,True,1,9946,0,21.023116,13.429605
3,S02,삼성전자,single_25,title_boundary,1,1.0,True,1.000000,1.0,True,...,True,1.0,5,5,False,0,1792,1792,23.805533,9.090649
4,S02,삼성전자,single_25,parent_child,1,1.0,True,1.000000,1.0,True,...,True,1.0,5,2,False,0,4989,2114,23.578350,6.536522
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,M_L,LG생활건강,multi_5,parent_child,2,2.0,True,0.500000,0.5,True,...,True,1.0,5,2,False,0,4407,1722,21.253468,7.997239
86,M_L,LG생활건강,multi_5,heading_parent_child,2,3.0,True,0.333333,0.5,True,...,False,0.0,4,0,True,4,317092,0,21.228524,92.964094
87,M_H,HDC,multi_5,title_boundary,2,8.0,False,0.000000,0.0,False,...,False,0.0,5,5,False,0,1643,1643,21.162402,8.960554
88,M_H,HDC,multi_5,parent_child,2,7.0,False,0.000000,0.0,False,...,False,0.0,5,2,False,0,5454,2318,20.901956,12.570727


## 10. 전체·질문 그룹·기업별 비교와 실패 확인

주요 판단은 **공통 예산 안에서 확보한 고정 근거 Recall**과 Hit입니다. 첫 근거 순위는 보조로 봅니다.
부모가 길다는 사실만으로 품질 우위를 주장하지 않습니다. 복합 질문은 5개뿐이라 방향 확인용이며 일반화할 수 없습니다.

In [13]:
summary = results.groupby('method').agg(
    questions=('id','count'), leaf_hit_at_5=('leaf_hit_at_5','mean'), leaf_mrr_at_5=('leaf_mrr_at_5','mean'),
    leaf_evidence_recall_at_5=('leaf_evidence_recall_at_5','mean'),
    context_hit_at_5=('context_hit_at_5','mean'), context_recall_at_5=('context_recall_at_5','mean'),
    budget_context_hit_at_5=('budget_context_hit_at_5','mean'), budget_context_recall_at_5=('budget_context_recall_at_5','mean'),
    median_raw_tokens=('raw_e5_tokens','median'), median_packed_tokens=('packed_e5_tokens','median'),
    median_search_ms=('search_ms','median'), p95_search_ms=('search_ms',lambda s:s.quantile(.95)),
    median_context_ms=('context_ms','median'),
    first_context_over_budget_rate=('first_context_over_budget','mean')).reindex(METHODS)
display(summary.T)
print('단일 25개와 복합 5개 결과 분리')
display(results.groupby(['dataset_group','method'])[['leaf_hit_at_5','leaf_evidence_recall_at_5',
    'budget_context_hit_at_5','budget_context_recall_at_5','raw_e5_tokens','packed_e5_tokens']].mean())
print('기업별 결과 — 기업당 6개 질문')
display(results.groupby(['company','method'])[['budget_context_hit_at_5','budget_context_recall_at_5']].mean())
print('동일 질문별 공통 예산 근거 Recall')
display(results.pivot(index=['company','id'], columns='method', values='budget_context_recall_at_5'))
print('부모 확장으로 회복했거나 예산 적용으로 근거가 빠진 사례')
changed = results[(results['context_recall_at_5'] != results['leaf_evidence_recall_at_5'])
                  | (results['budget_context_recall_at_5'] != results['context_recall_at_5'])]
display(changed[['id','method','leaf_evidence_recall_at_5','context_recall_at_5','budget_context_recall_at_5','raw_e5_tokens','packed_e5_tokens']])
print('큰 실제 제목 부모가 공통 예산에 들어가지 못한 사례 — 검색 실패와 구분')
display(results.loc[results['first_context_over_budget'], ['id','method','raw_e5_tokens','packed_e5_tokens','context_recall_at_5','budget_context_recall_at_5']])
failed = results.loc[results['budget_context_recall_at_5'] < 1, ['id','method']]
with pd.option_context('display.max_colwidth', None, 'display.max_rows', 90):
    display(failed.merge(detail_df, on=['id','method']))


method,title_boundary,parent_child,heading_parent_child
questions,30.000000,30.000000,30.000000
leaf_hit_at_5,0.866667,0.800000,0.866667
leaf_mrr_at_5,0.547778,0.574444,0.564444
leaf_evidence_recall_at_5,0.833333,0.766667,0.833333
context_hit_at_5,0.866667,0.833333,1.000000
context_recall_at_5,0.833333,0.816667,1.000000
budget_context_hit_at_5,0.866667,0.766667,0.000000
budget_context_recall_at_5,0.833333,0.750000,0.000000
median_raw_tokens,1691.000000,4276.000000,36889.000000
median_packed_tokens,1691.000000,2052.000000,0.000000


단일 25개와 복합 5개 결과 분리


leaf_hit_at_5  leaf_evidence_recall_at_5  \
dataset_group method                                                           
multi_5       heading_parent_child           0.80                       0.60   
              parent_child                   0.80                       0.60   
              title_boundary                 0.80                       0.60   
single_25     heading_parent_child           0.88                       0.88   
              parent_child                   0.80                       0.80   
              title_boundary                 0.88                       0.88   

                                    budget_context_hit_at_5  \
dataset_group method                                          
multi_5       heading_parent_child                     0.00   
              parent_child                             0.80   
              title_boundary                           0.80   
single_25     heading_parent_child                     0.00   
              parent_child                             0.76   
              title_boundary                           0.88   

                                    budget_context_recall_at_5  raw_e5_tokens  \
dataset_group method                                                            
multi_5       heading_parent_child                        0.00      271009.40   
              parent_child                                0.70        4123.60   
              title_boundary                              0.60        1734.60   
single_25     heading_parent_child                        0.00      180589.04   
              parent_child                                0.76        4111.52   
              title_boundary                              0.88        1669.04   

                                    packed_e5_tokens  
dataset_group method                                  
multi_5       heading_parent_child              0.00  
              parent_child                   1960.80  
              title_boundary                 1734.60  
single_25     heading_parent_child              0.00  
              parent_child                   1976.24  
              title_boundary                 1669.04

기업별 결과 — 기업당 6개 질문


budget_context_hit_at_5  \
company method                                          
HDC     heading_parent_child                 0.000000   
        parent_child                         0.500000   
        title_boundary                       0.666667   
LG생활건강  heading_parent_child                 0.000000   
        parent_child                         0.666667   
        title_boundary                       0.833333   
NAVER   heading_parent_child                 0.000000   
        parent_child                         1.000000   
        title_boundary                       1.000000   
리노공업    heading_parent_child                 0.000000   
        parent_child                         0.666667   
        title_boundary                       0.833333   
삼성전자    heading_parent_child                 0.000000   
        parent_child                         1.000000   
        title_boundary                       1.000000   

                              budget_context_recall_at_5  
company method                                            
HDC     heading_parent_child                    0.000000  
        parent_child                            0.500000  
        title_boundary                          0.666667  
LG생활건강  heading_parent_child                    0.000000  
        parent_child                            0.666667  
        title_boundary                          0.750000  
NAVER   heading_parent_child                    0.000000  
        parent_child                            1.000000  
        title_boundary                          1.000000  
리노공업    heading_parent_child                    0.000000  
        parent_child                            0.583333  
        title_boundary                          0.750000  
삼성전자    heading_parent_child                    0.000000  
        parent_child                            1.000000  
        title_boundary                          1.000000

...

부모 확장으로 회복했거나 예산 적용으로 근거가 빠진 사례


,id,method,leaf_evidence_recall_at_5,context_recall_at_5,budget_context_recall_at_5,raw_e5_tokens,packed_e5_tokens
2,S01,heading_parent_child,1.0,1.0,0.0,9946,0
5,S02,heading_parent_child,1.0,1.0,0.0,23660,0
8,S03,heading_parent_child,1.0,1.0,0.0,23874,0
11,S04,heading_parent_child,1.0,1.0,0.0,23874,0
14,S05,heading_parent_child,1.0,1.0,0.0,23660,0
17,R01,heading_parent_child,0.0,1.0,0.0,15429,0
20,R02,heading_parent_child,1.0,1.0,0.0,63766,0
23,R03,heading_parent_child,1.0,1.0,0.0,42902,0
25,R04,parent_child,0.0,1.0,1.0,2893,2357
26,R04,heading_parent_child,1.0,1.0,0.0,42902,0


큰 실제 제목 부모가 공통 예산에 들어가지 못한 사례 — 검색 실패와 구분


,id,method,raw_e5_tokens,packed_e5_tokens,context_recall_at_5,budget_context_recall_at_5
2,S01,heading_parent_child,9946,0,1.0,0.0
5,S02,heading_parent_child,23660,0,1.0,0.0
8,S03,heading_parent_child,23874,0,1.0,0.0
11,S04,heading_parent_child,23874,0,1.0,0.0
14,S05,heading_parent_child,23660,0,1.0,0.0
17,R01,heading_parent_child,15429,0,1.0,0.0
20,R02,heading_parent_child,63766,0,1.0,0.0
23,R03,heading_parent_child,42902,0,1.0,0.0
26,R04,heading_parent_child,42902,0,1.0,0.0
29,R05,heading_parent_child,25713,0,1.0,0.0


,id,method,company,question,missing_reference_ids,reference_quotes,top1_leaf,first_context_preview
0,S01,heading_parent_child,삼성전자,삼성전자가 주식을 일반에 공개한 시점은 언제인가?,[S01],[1975년 6월 11일 기업공개를 실시하였습니다.],"기타 | - - - -\nⅣ. 발행주식의 총수 (Ⅱ-Ⅲ) | 5,969,782,550 822,886,700 6,792,669,250 -\nⅤ. 자기주식수 | 29,700,000 4,050,000 33,750,000 -\nⅥ. 유통주식수 (Ⅳ-Ⅴ) | 5,940,082,550 818,836,700 6,758,919,250 -\nⅦ. 자기주식 보유비율 | 0.5 0.5 0.5 - 나 . 자기주식 당사는 2024년 중 주주가치 제고의 목적","## I. 회사의 개요\n\n## 1. 회사의 개요\n가. 회사의 법적ㆍ상업적 명칭 당사의 명칭은 삼성전자주식회사이고 영문명은 Samsung Electronics Co., Ltd. 입니다. 나. 설립일자 당사는 1969년 1월 13일에 삼성전자공업주식회사로 설립되었으며, 1975년 6월 11일 기업공개를 실시하였습니다. 당사는 1984년 2월 28일 정기주주총회 결의에 의거하여 상호를 삼성전자공업주식회사에서 삼성전자주식회사로 변경하였습니다. 다. 본사의 주소, 전화번호 및 홈페이지 주소: 경기도 수원시 영통구 삼성로 129(매탄동) 전화번호: 031-200-1114 홈페이"
1,S02,heading_parent_child,삼성전자,삼성전자공업에서 지금의 회사명으로 바뀐 날짜는?,[S02],[1984년 2월 28일 정기주주총회 결의에 의거하여 상호를],"## 1. 회사의 개요\n가. 회사의 법적ㆍ상업적 명칭 당사의 명칭은 삼성전자주식회사이고 영문명은 Samsung Electronics Co., Ltd. 입니다. 나. 설립일자 당사는 1969년 1월 13일에 삼성전자공업주식회사로 설립되었으며, 1975년 6월 11일 기업공개를 실시하였습니다. 당사는 1984년 2월 28일 정기주주총회 결의에 의거하여 상호를 삼성전자공업주식회사에서 삼성전자주식회사로 변경하였습니다. 다. 본사의 주","## I. 회사의 개요\n\n## 1. 회사의 개요\n가. 회사의 법적ㆍ상업적 명칭 당사의 명칭은 삼성전자주식회사이고 영문명은 Samsung Electronics Co., Ltd. 입니다. 나. 설립일자 당사는 1969년 1월 13일에 삼성전자공업주식회사로 설립되었으며, 1975년 6월 11일 기업공개를 실시하였습니다. 당사는 1984년 2월 28일 정기주주총회 결의에 의거하여 상호를 삼성전자공업주식회사에서 삼성전자주식회사로 변경하였습니다. 다. 본사의 주소, 전화번호 및 홈페이지 주소: 경기도 수원시 영통구 삼성로 129(매탄동) 전화번호: 031-200-1114 홈페이"
2,S03,heading_parent_child,삼성전자,삼성전자 본사의 소재지는 어디인가?,[S03],[경기도 수원시 영통구 삼성로 129(매탄동)],"## 1. 회사의 개요\n가. 회사의 법적ㆍ상업적 명칭 당사의 명칭은 삼성전자주식회사이고 영문명은 Samsung Electronics Co., Ltd. 입니다. 나. 설립일자 당사는 1969년 1월 13일에 삼성전자공업주식회사로 설립되었으며, 1975년 6월 11일 기업공개를 실시하였습니다. 당사는 1984년 2월 28일 정기주주총회 결의에 의거하여 상호를 삼성전자공업주식회사에서 삼성전자주식회사로 변경하였습니다. 다. 본사의 주","## I. 회사의 개요\n\n## 1. 회사의 개요\n가. 회사의 법적ㆍ상업적 명칭 당사의 명칭은 삼성전자주식회사이고 영문명은 Samsung Electronics Co., Ltd. 입니다. 나. 설립일자 당사는 1969년 1월 13일에 삼성전자공업주식회사로 설립되었으며, 1975년 6월 11일 기업공개를 실시하였습니다. 당사는 1984년 2월 28일 정기주주총회 결의에 의거하여 상호를 삼성전자공업주식회사에서 삼성전자주식회사로 변경하였습니다. 다. 본사의 주소, 전화번호 및 홈페이지 주소: 경기도 수원시 영통구 삼성로 129(매탄동) 전화번호: 031-200-1114 홈페이"
3,S04,heading_parent_child,삼성전자,삼성전자 반도체 사업 부문은 어떤 제품을 주로 만드는가?,[S04],"[DS 부문 | DRAM, NAND Flash, 모바일AP 등]","## 1. 사업의 개요\n당사는 본사를 거점으로 한국과 DX 부문 산하 해외 9개 지역총괄 및 DS 부문 산하 해외 5개 지역총괄의 생산ㆍ판매법인, SDC 및 Harman 산하 종속기업 등 228개의 종속기업으로 구성된 글로벌 전자 기업입니다. 사업별로 보면, Set 사업은 DX(Device eXperience) 부문이 TV를 비롯하여 모니터, 냉장고, 세탁기, 에어컨, 스마트폰, 네트워크시스템, PC 등을 생산ㆍ판매하며, 부품","## II. 사업의 내용\n\n## 1. 사업의 개요\n당사는 본사를 거점으로 한국과 DX 부문 산하 해외 9개 지역총괄 및 DS 부문 산하 해외 5개 지역총괄의 생산ㆍ판매법인, SDC 및 Harman 산하 종속기업 등 228개의 종속기업으로 구성된 글로벌 전자 기업입니다. 사업별로 보면, Set 사업은 DX(Device eXperience) 부문이 TV를 비롯하여 모니터, 냉장고, 세탁기, 에어컨, 스마트폰, 네트워크시스템, PC 등을 생산ㆍ판매하며, 부품 사업은 DS(Device Solutions) 부문에서 DRAM, NAND Flash, 모바일AP 등의 제품을 생산ㆍ"
4,S05,heading_parent_child,삼성전자,삼성전자 전장 사업에 해당하는 하만의 주요 제품은?,[S05],"[Harman | 디지털 콕핏(Digital Cockpit), 카오디오, 포터블 스피커 등]","## 1. 회사의 개요\n가. 회사의 법적ㆍ상업적 명칭 당사의 명칭은 삼성전자주식회사이고 영문명은 Samsung Electronics Co., Ltd. 입니다. 나. 설립일자 당사는 1969년 1월 13일에 삼성전자공업주식회사로 설립되었으며, 1975년 6월 11일 기업공개를 실시하였습니다. 당사는 1984년 2월 28일 정기주주총회 결의에 의거하여 상호를 삼성전자공업주식회사에서 삼성전자주식회사로 변경하였습니다. 다. 본사의 주","## I. 회사의 개요\n\n## 1. 회사의 개요\n가. 회사의 법적ㆍ상업적 명칭 당사의 명칭은 삼성전자주식회사이고 영문명은 Samsung Electronics Co., Ltd. 입니다. 나. 설립일자 당사는 1969년 1월 13일에 삼성전자공업주식회사로 설립되었으며, 1975년 6월 11일 기업공개를 실시하였습니다. 당사는 1984년 2월 28일 정기주주총회 결의에 의거하여 상호를 삼성전자공업주식회사에서 삼성전자주식회사로 변경하였습니다. 다. 본사의 주소, 전화번호 및 홈페이지 주소: 경기도 수원시 영통구 삼성로 129(매탄동) 전화번호: 031-200-1114 홈페이"
5,R01,title_boundary,리노공업,리노공업은 납품 단가를 정할 때 고객 주문에서 무엇을 살피는가?,[R01],[거래처별 공급수량 및 제품의 난이도에 따라 합리적으로 가격을 책정],"아울러 반도체 시장의 변화에 따른 원가 절감 및 성능 향상을 위한 공정개선, 즉 반도체 패키지 공정 축소 및 개선에 대한 기술변화를 선지하여 이에 적합한 제품을 출시할 수 있도록 연구개발 인력을 투입하고 있으며, 신규산업분야에 진출하기 위한 제품 및 공정연구도 함께 하고 있습니다. 주요 연구개발 실적과 계획은 아래와 같습니다.1) 주요 연구개발 실적\n연구과제 | 저저항/저하중스프링핀개발 |\n연구기관 | 리노공업 기술연구소 |\n","아울러 반도체 시장의 변화에 따른 원가 절감 및 성능 향상을 위한 공정개선, 즉 반도체 패키지 공정 축소 및 개선에 대한 기술변화를 선지하여 이에 적합한 제품을 출시할 수 있도록 연구개발 인력을 투입하고 있으며, 신규산업분야에 진출하기 위한 제품 및 공정연구도 함께 하고 있습니다. 주요 연구개발 실적과 계획은 아래와 같습니다.1

## 11. 재현·자원 기록과 해석 기준

임베딩 파라미터 메모리·벡터 배열·텍스트 크기·SQLite 페이지 크기는 전체 RAM peak가 아닙니다.
검색 시간은 인덱스 준비 뒤의 로컬 구성 시간이며 최종 사용자 응답 시간은 아닙니다. 부모 확장의 LLM 비용은 호출하지 않아 측정하지 않습니다.
기록된 E5-token은 향후 운영 모델의 토크나이저로 다시 확인해야 합니다.

In [14]:
runtime = {'observed_at_utc':datetime.now(timezone.utc).isoformat(), 'dataset_type':'development',
    'companies':len(REPORTS),'reports':len(REPORTS),'questions':len(qcases), 'single_questions':25,'multi_questions':5,
    'evaluation_sha256':evaluation_sha256,'reused_ast_sha256':reuse_sha256,
    'embedding_model':MODEL_ID,'model_revision':MODEL_REVISION,'device':'cpu',
    'methods':list(METHODS), 'heading_policy':'XML SECTION/TITLE/SUBTITLE; original parent; split search body only if E5 limit exceeded',
    'vector_bm25_weights':[0.5,0.5],'candidate_depth_each':CANDIDATE_DEPTH,'rrf_k':RRF_K,
    'child_tokens':CHILD_SIZE,'parent_tokens':PARENT_SIZE,'overlap':OVERLAP,'retrieved_leaves':EVAL_K,
    'context_budget_e5_tokens':CONTEXT_BUDGET,'context_packing':'whole units, ranked prefix, stop before overflow',
    'model_parameter_MiB':parameter_MiB,'model_load_seconds':model_load_seconds,
    'llm_calls':0,'openai_embedding_calls':0,
    'packages':{p:importlib.metadata.version(p) for p in ['llama-index-core','transformers','torch','kiwipiepy']}}
print(json.dumps(runtime,ensure_ascii=False,indent=2))
display(chunk_info)
display(pd.DataFrame(bm25_rows)); display(pd.DataFrame(e5_rows))
print('외부 LLM/임베딩 API 과금 없음. 로컬 CPU·RAM·전력, OpenDART 사용 제한은 별도입니다.')
print('최종 청킹·운영 구조는 자동 채택하지 않습니다. 저장 후 결과를 함께 해석합니다.')


{
  "observed_at_utc": "2026-10-05T16:10:43.963678+00:00",
  "dataset_type": "development",
  "companies": 5,
  "reports": 5,
  "questions": 30,
  "single_questions": 25,
  "multi_questions": 5,
  "evaluation_sha256": "a8bbc6202b3b0956c7519660a6e760eb1dab9be1d808118bdba1e7557c3c6ebe",
  "reused_ast_sha256": "0c51db7110a13828d60f8a92999c33ef19661bc0dc5d7fce16c68862082c3eab",
  "embedding_model": "intfloat/multilingual-e5-small",
  "model_revision": "5697a65b0a002a92fe8c4fc9d495303ffff9c7d2",
  "device": "cpu",
  "methods": [
    "title_boundary",
    "parent_child",
    "heading_parent_child"
  ],
  "heading_policy": "XML SECTION/TITLE/SUBTITLE; original parent; split search body only if E5 limit exceeded",
  "vector_bm25_weights": [
    0.5,
    0.5
  ],
  "candidate_depth_each": 20,
  "rrf_k": 60,
  "child_tokens": 400,
  "parent_tokens": 1200,
  "overlap": 48,
  "retrieved_leaves": 5,
  "context_budget_e5_tokens": 2400,
  "context_packing": "whole units, ranked prefix, stop before ov

,method,company,search_chunks,return_units,max_e5_tokens,chunking_seconds,search_text_MiB,context_text_MiB
0,title_boundary,삼성전자,82,82,403,0.252669,0.082886,0.082886
1,title_boundary,리노공업,333,333,404,0.536452,0.309031,0.309031
2,title_boundary,NAVER,1168,1168,404,2.871409,1.146790,1.146790
3,title_boundary,LG생활건강,824,824,404,1.725481,0.811188,0.811188
4,title_boundary,HDC,123,123,404,0.240656,0.115170,0.115170
5,parent_child,삼성전자,91,32,403,0.223959,0.084018,0.080358
6,parent_child,리노공업,349,146,404,0.797465,0.309011,0.299274
7,parent_child,NAVER,1293,418,404,4.325515,1.153458,1.096161
8,parent_child,LG생활건강,902,325,404,2.824909,0.815345,0.773008
9,parent_child,HDC,134,46,399,0.389932,0.115545,0.110558


,method,company,bm25_build_seconds,sqlite_MiB,empty_bm25_chunks
0,title_boundary,삼성전자,1.781508,0.140625,0
1,title_boundary,리노공업,1.487323,0.421875,0
2,title_boundary,NAVER,4.737773,1.476562,1
3,title_boundary,LG생활건강,3.815310,1.031250,0
4,title_boundary,HDC,0.516250,0.179688,0
5,parent_child,삼성전자,0.397734,0.140625,0
6,parent_child,리노공업,1.609407,0.417969,0
7,parent_child,NAVER,4.833820,1.476562,2
8,parent_child,LG생활건강,4.189770,1.027344,0
9,parent_child,HDC,0.549753,0.183594,0


,method,company,new_embedding_texts,reused_records,e5_build_seconds,vectors_MiB
0,title_boundary,삼성전자,82,0,3.212246,0.120117
1,title_boundary,리노공업,331,2,12.674248,0.487793
2,title_boundary,NAVER,1156,12,45.182544,1.710938
3,title_boundary,LG생활건강,809,15,31.531195,1.207031
4,title_boundary,HDC,120,3,4.540928,0.180176
5,parent_child,삼성전자,32,59,1.170317,0.133301
6,parent_child,리노공업,84,265,3.251469,0.511230
7,parent_child,NAVER,668,625,27.357229,1.894043
8,parent_child,LG생활건강,395,507,15.144419,1.321289
9,parent_child,HDC,73,61,2.845222,0.196289


외부 LLM/임베딩 API 과금 없음. 로컬 CPU·RAM·전력, OpenDART 사용 제한은 별도입니다.
최종 청킹·운영 구조는 자동 채택하지 않습니다. 저장 후 결과를 함께 해석합니다.


## 결과를 읽을 때

- B의 leaf 검색이 좋아졌다면 계층 분할로 검색 청크 경계가 바뀐 효과를 포함합니다.
- C는 XML 제목 계층 후보이며 B와 달리 부모 크기를 토큰으로 정하지 않습니다. C에서 예산 점수만 낮고 원래 문맥 점수가 높으면 큰 부모를 담지 못한 정책 영향인지 확인합니다. 부모를 창문 단위로 조립하는 개선은 이번에 섞지 않습니다.
- leaf는 비슷한데 반환 문맥 Recall이 좋아졌다면 부모 확장이 근거를 함께 가져온 효과입니다.
- 무제한 문맥에서만 좋아지고 같은 예산에서 좋아지지 않으면 길이 증가와 품질 향상을 구분해야 합니다.
- 지표가 같아도 주변 설명·표 제목을 실제로 더 잘 활용하는지는 **후속 LLM 답변 평가**가 필요합니다.
- 이번 30개 개발 질문만으로 청킹 최적 크기·최종 검색·답변 품질·할루시네이션을 확정하지 않습니다.

### 근거 자료 — 2026-10-05 확인

- [LlamaIndex HierarchicalNodeParser 공식 API](https://developers.llamaindex.ai/python/framework-api-reference/node_parsers/hierarchical/): 큰 부모와 작은 자식 Node 관계를 생성합니다.
- [LlamaIndex Auto Merging 공식 예제](https://developers.llamaindex.ai/python/framework/integrations/retrievers/auto_merging_retriever/): leaf를 인덱싱하고 부모 문맥을 별도로 보존하는 구조의 근거입니다. 임계값 병합 클래스 자체는 이번에 사용하지 않습니다.
- [29번 notebook](29_rrf_weight_comparison.ipynb): 데이터·원문 선택·검색 가중치 기준선.
- [평가 지표 정의](../../docs/EVALUATION_METRICS.md): 반환 문맥 단계의 Hit·근거 Recall과 토큰 예산을 구분합니다.